In [1]:
def foo(a,b,c,d=0):
    return f"a={a}, b={b}, c={c}, d={d}"


In [2]:
from inspect import signature
from inspect import Parameter as prm


class Wrapper:
    """
    Classe per wrappare una funzione, congelando alcuni parametri e consentendo
    la chiamata con i parametri non congelati in qualsiasi ordine.
    """

    def __init__(self, func, **frozen_params):
        """
        Inizializza la funzione wrappata e congela i parametri forniti.

        Args:
            func: La funzione originale da wrappare.
            frozen_params: I parametri da congelare e i loro valori.
        """
        self._func = func
        self.sig = signature(func)
        #NOTE sostituire all params con mio param
        self.all_params = list(self.sig.parameters.values())

        # Verifica che i parametri congelati siano validi
        for param in frozen_params:
            if param not in self.sig.parameters:
                raise ValueError(
                    f"Parametro '{param}' non esiste nella funzione '{func.__name__}'"
                )

        self.frozen_params = dict(frozen_params)  # Copia mutabile
        self._rebuild_signature()
    
    @property
    def func(self):
        '''funzione è proprietà immutabile del wrapper'''
        return self._func
    
    

    
    def _rebuild_signature(self):
        """
        Ricostruisce la firma della funzione wrappata dopo eventuali modifiche ai default.
        Mantiene i parametri congelati in fondo, ma riordina i parametri in modo che
        quelli senza default vengano prima di quelli con default.
        """
        # Parametri non congelati
        self.unfrozen_params = [
            p for p in self.all_params if p.name not in self.frozen_params
        ]

        # Parametri congelati aggiornati con i valori di default attuali
        self.frozen_params_list = [
            p.replace(default=self.frozen_params[p.name])
            for p in self.all_params
            if p.name in self.frozen_params
        ]

        # Separiamo i parametri non congelati in base al default
        no_default_unfrozen = [
            p for p in self.unfrozen_params if p.default is prm.empty
        ]
        default_unfrozen = [
            p for p in self.unfrozen_params if p.default is not prm.empty
        ]

        # Ordine: unfrozen senza default, unfrozen con default, frozen (che hanno default)
        new_params = no_default_unfrozen + default_unfrozen + self.frozen_params_list
        self.new_sig = self.sig.replace(parameters=new_params)

    def __call__(self, *args, **kwargs):
        """
        Chiama la funzione wrappata con gli argomenti forniti.

        Args:
            *args: Argomenti posizionali per i parametri non congelati.
            **kwargs: Argomenti con nome per i parametri non congelati.

        Returns:
            Il risultato della funzione originale con i parametri congelati.
        """
        # Associa gli argomenti alla nuova firma (partial per permettere flessibilità)
        bound_args = self.new_sig.bind_partial(*args, **kwargs)
        # Applica i default per i parametri non passati
        bound_args.apply_defaults()

        # Ricostruisci gli argomenti per la funzione originale
        final_args = {name: bound_args.arguments[name] for name in self.sig.parameters}

        return self.func(**final_args)

    def get_defaults(self):
        """
        Restituisce un dizionario con i default attuali di tutti i parametri,
        inclusi quelli congelati e non.
        """
        defaults = {}
        for p in self.all_params:
            if p.name in self.frozen_params:
                defaults[p.name] = self.frozen_params[p.name]
            else:
                if p.default is not prm.empty:
                    defaults[p.name] = p.default
                else:
                    defaults[p.name] = None
        return defaults

    def set_default(self, param_name, new_value):
        """
        Imposta un nuovo valore di default per il parametro indicato.

        Se il parametro è congelato, aggiorna il valore congelato.
        Se non è congelato, aggiorna il default del parametro.
        Infine, rigenera la firma.
        """
        if param_name not in self.sig.parameters:
            raise ValueError(
                f"Parametro '{param_name}' non esiste nella funzione '{self.func.__name__}'"
            )

        updated_params = []
        for p in self.all_params:
            if p.name == param_name:
                p = p.replace(default=new_value)
            updated_params.append(p)
        self.all_params = updated_params

        #if param_name not in self.frozen_params:
        self.frozen_params[param_name] = new_value            
        self._rebuild_signature()

    def unfreeze_param(self, param_name):
        """
        Rimuove un parametro dalla lista dei congelati, rendendolo nuovamente unfrozen.
        """
        if param_name not in self.frozen_params:
            raise ValueError(f"Parametro '{param_name}' non è attualmente congelato.")
        
        self.frozen_params.pop(param_name)
        self._rebuild_signature()
    


wrapped = Wrapper(foo, a=1, c=2)

print(wrapped(11,22))

wrapped.set_default('b',33)
wrapped.set_default('b', 23)
#wrapped = Wrapper(foo, b=10)
print(wrapped(11))  # Restituisce 1+10+99 = 110

# Ora scongelo b:
wrapped.unfreeze_param("b")
print(
    wrapped(11, 22)
)  # Ora b è unfrozen, posso passarlo come argomento normale (1+2+99 = 102)

print(wrapped.func(1,2,3,4))
print(wrapped.sig)

a=1, b=11, c=2, d=22
a=1, b=23, c=2, d=11
a=1, b=11, c=2, d=22
a=1, b=2, c=3, d=4
(a, b, c, d=0)


In [19]:
from inspect import signature

class FreezableFunction:
    """
    Classe per wrappare una funzione, congelando alcuni dei suoi parametri.
    
    Args:
        func: La funzione originale da wrappare.
        **frozen_params: I parametri da congelare e i loro valori.
    """
    def __init__(self, func, **frozen_params):
        self.func = func
        self.sig = signature(func)
        self.all_params = list(self.sig.parameters.keys())
        
        # Verifica che i parametri congelati esistano nella funzione originale
        for param in frozen_params:
            if param not in self.all_params:
                raise ValueError(f"Parametro '{param}' non esiste nella funzione '{func.__name__}'")
        
        self.frozen_params = frozen_params
        # Parametri non congelati
        self.unfrozen_params = [p for p in self.all_params if p not in frozen_params]
        
        
    
    def __call__(self, *args, **kwargs):
        """
        Chiama la funzione wrappata con gli argomenti forniti.
        
        Args:
            *args: Valori per i parametri non congelati, forniti in ordine.
        
        Returns:
            Il risultato della funzione originale con i parametri congelati.
        """
        if len(args) > len(self.unfrozen_params):
            raise ValueError(f"Troppi argomenti forniti. Aspettati al massimo {len(self.unfrozen_params)}.")
        
        # Mappa gli args sui parametri non congelati
        provided_args = dict(zip(self.unfrozen_params, args))
        
        # Combina i parametri forniti con quelli congelati
        final_args = {**self.frozen_params, **provided_args, **kwargs}
        
        # Ordina gli argomenti secondo l'ordine della funzione originale
        #ordered_args = [final_args[param] for param in self.all_params]
        #print('chiamata con,', final_args)
        
        # Chiama la funzione originale
        return self.func(**final_args)



# Wrappiamo la funzione congelando 'b' e 'd'
wrapped_class = FreezableFunction(foo, a=1,d=2)
wrapped_class.frozen_params['a'] = 22
# Chiamiamo la funzione fornendo solo gli argomenti non congelati come args
print(wrapped_class(1, 30, a=2.3))  # a=1, b=10, c=30, d=20
print(wrapped_class(5, 15, b= 0.2))  # a=5, b=10, c=15, d=20
print(wrapped_class(1,1))


a=2.3, b=1, c=30, d=2
a=22, b=0.2, c=15, d=2
a=22, b=1, c=1, d=2


In [15]:
def class1():
    return wrapped(1,2)

def class2():
    return wrapped_class(1,2)


import timeit

time_class1 = timeit.timeit(class1, number=100_000)
time_class2 = timeit.timeit(class2, number=100_000)


print(f"Tempo funzione eval: {time_class1:.6f} secondi")
print(f"tempo wrapped {time_class2}")


Tempo funzione eval: 1.205623 secondi
tempo wrapped 0.1680854480000562


In [6]:
from inspect import signature
from inspect import signature, Parameter


def merge_functions_multiple(func, num_calls):
    """
    Unisce più chiamate della stessa funzione in una nuova funzione.

    Args:
        func: La funzione da ripetere.
        num_calls: Il numero di volte che la funzione deve essere chiamata.

    Returns:
        Una nuova funzione che combina i risultati di `num_calls` chiamate.
    """
    sig = signature(func)
    original_params = list(sig.parameters.values())

    # Creiamo i nuovi parametri con suffissi per ogni chiamata
    new_params = []
    for i in range(1, num_calls + 1):
        for param in original_params:
            new_params.append(
                Parameter(
                    name=f"{param.name}_{i}", kind=param.kind, default=param.default
                )
            )

    # Creiamo una nuova firma
    combined_signature = sig.replace(parameters=new_params)

    def combined_function(*args, **kwargs):
        results = []
        # Processiamo ogni chiamata separatamente
        for i in range(1, num_calls + 1):
            call_args = {
                param.name: kwargs.pop(f"{param.name}_{i}", None)
                for param in original_params
                if f"{param.name}_{i}" in kwargs
            }

            # Aggiungiamo i valori posizionali in sequenza
            call_args.update(
                zip(
                    [param.name for param in original_params],
                    args[(i - 1) * len(original_params) : i * len(original_params)],
                )
            )

            # Chiamata alla funzione originale
            results.append(func(**call_args))

        # Combiniamo i risultati
        return sum(results)  # Puoi cambiare la combinazione qui

    # Aggiorniamo la firma della funzione combinata
    combined_function.__signature__ = combined_signature
    return combined_function


def foo(a, b):
    return a + b


# Combiniamo la funzione `foo` per due chiamate
combined = merge_functions_multiple(foo, 2)

# La nuova funzione combinata accetta parametri con suffissi
print(combined(1, 2, 3, 4))  # Output: (1 + 2) + (3 + 4) = 10
print(combined(a_1=5, b_1=6, a_2=2, b_2=3))  # Output: (5 + 6) + (2 + 3) = 16

test= merge_functions_multiple(combined, 2)
print(test(1,1,1,1,1,1,1,1,1))


10
16
8
